# Prueba técnica Insights
## Ejercicio 1 – Manejo de datos

**Santiago Osorio Gómez**  
**Cédula:** 1000.099.104  
**Fecha:** 12 de marzo de 2026

In [2]:
from pathlib import Path
import pandas as pd

# ============================================================
# CONFIGURACION DEL PROYECTO
# ============================================================

def find_project_root() -> Path:
    """
    Busca la raiz del proyecto subiendo desde el directorio actual
    hasta encontrar la estructura esperada.
    """
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    required_dirs = {"data", "docs", "notebooks", "outputs", "src"}

    for candidate in candidates:
        if candidate.exists():
            dir_names = {p.name for p in candidate.iterdir() if p.is_dir()}
            if required_dirs.issubset(dir_names):
                return candidate

    raise FileNotFoundError(
        "No se encontro la raiz del proyecto. "
        "Asegurate de ejecutar esto dentro de ~/Documents/prueba_insights"
    )

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
DOCS_DIR = ROOT / "docs"
OUTPUTS_DIR = ROOT / "outputs"
EJ1_OUTPUT_DIR = OUTPUTS_DIR / "ejercicio_1"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
EJ1_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raiz del proyecto detectada: {ROOT}")
print(f"Directorio de datos: {DATA_DIR}")
print(f"Directorio docs: {DOCS_DIR}")
print(f"Directorio outputs ejercicio 1: {EJ1_OUTPUT_DIR}")

# ============================================================
# VALIDACION DE ARCHIVOS REALES DEL PROYECTO
# ============================================================

required_files = {
    "instrucciones_pdf": DATA_DIR / "instrucciones.pdf",
    "prueba_xlsx": DATA_DIR / "prueba.xlsx",
    "withdrawals_xlsx": DATA_DIR / "withdrawals.xlsx",
}

print("\nVerificacion de archivos reales del proyecto:")
for name, path in required_files.items():
    print(f"- {name}: {'OK' if path.exists() else 'FALTA'} -> {path}")

missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan archivos requeridos en data/:\n" + "\n".join(missing)
    )

# ============================================================
# HELPERS
# ============================================================

def show_df(name: str, df: pd.DataFrame, max_rows: int = 10) -> None:
    print(f"\n{name}")
    print("-" * len(name))
    try:
        from IPython.display import display
        display(df.head(max_rows))
    except Exception:
        print(df.head(max_rows).to_string(index=False))

def filter_table(df: pd.DataFrame, field: str, value):
    """
    Filtra una tabla por:
    - igualdad simple
    - lista de valores
    - rango de fechas o rango numerico si value tiene 2 elementos
    """
    out = df.copy()

    if isinstance(value, (list, tuple, set)) and not isinstance(value, str):
        value = list(value)

        if len(value) == 2 and field in out.columns:
            col = out[field]

            if pd.api.types.is_datetime64_any_dtype(col):
                start = pd.to_datetime(value[0])
                end = pd.to_datetime(value[1])
                return out[(col >= start) & (col <= end)].copy()

            if pd.api.types.is_numeric_dtype(col):
                return out[(col >= value[0]) & (col <= value[1])].copy()

        return out[out[field].isin(value)].copy()

    return out[out[field] == value].copy()

def join_tables(from_table: pd.DataFrame, to_table: pd.DataFrame, fields):
    """
    Join inner usando una o varias llaves.
    """
    if isinstance(fields, str):
        fields = [fields]
    return from_table.merge(to_table, on=fields, how="inner")

def drop_fields(df: pd.DataFrame, fields):
    """
    Elimina columnas si existen.
    """
    out = df.copy()
    if isinstance(fields, str):
        fields = [fields]
    fields = [f for f in fields if f in out.columns]
    return out.drop(columns=fields)

# ============================================================
# TABLAS LITERALES DEL PDF
# ============================================================

usuarios_literal = pd.DataFrame({
    "id_usuario": [123455, 123456],
    "email": ["analista@insightswm.com", "inversiones@insightswm.com"],
    "id_asesor": ["#1358", "#135813"],
    "perfil_riesgo": [5, 7],
})

portafolios_literal = pd.DataFrame({
    "id_portafolio": [45678, 45789],
    "id_usuario": [123455, 123456],
    "tipo_portafolio": ["acciones", "bonos"],
})

asesores_literal = pd.DataFrame({
    "id_asesor": [9876, 9875],
    "email": ["analista@insightswm.com", "inversiones@insightswm.com"],
    "num_clientes": [3, 12],
})

balances_literal = pd.DataFrame({
    "id_portafolio": [10123, 10124],
    "fecha": pd.to_datetime(["2022-10-12", "2022-10-12"]),
    "balance": [1500.15, 54300.60],
})

show_df("Usuarios - version literal PDF", usuarios_literal)
show_df("Portafolios - version literal PDF", portafolios_literal)
show_df("Asesores - version literal PDF", asesores_literal)
show_df("Balances - version literal PDF", balances_literal)

# Guardar tablas literales
usuarios_literal.to_csv(EJ1_OUTPUT_DIR / "usuarios_literal.csv", index=False)
portafolios_literal.to_csv(EJ1_OUTPUT_DIR / "portafolios_literal.csv", index=False)
asesores_literal.to_csv(EJ1_OUTPUT_DIR / "asesores_literal.csv", index=False)
balances_literal.to_csv(EJ1_OUTPUT_DIR / "balances_literal.csv", index=False)

# ============================================================
# CHEQUEO DE CONSISTENCIA DEL EJEMPLO LITERAL
# ============================================================

ids_usuarios_literal = set(usuarios_literal["id_asesor"].astype(str))
ids_asesores_literal = set(asesores_literal["id_asesor"].astype(str))
ids_portafolio_literal = set(portafolios_literal["id_portafolio"].astype(int))
ids_balance_literal = set(balances_literal["id_portafolio"].astype(int))

advisor_id_matches = ids_usuarios_literal.intersection(ids_asesores_literal)
portfolio_id_matches = ids_portafolio_literal.intersection(ids_balance_literal)

print("\nChequeo de consistencia del ejemplo literal del PDF:")
print(f"- Coincidencias entre Usuarios.id_asesor y Asesores.id_asesor: {advisor_id_matches if advisor_id_matches else 'NINGUNA'}")
print(f"- Coincidencias entre Portafolios.id_portafolio y Balances.id_portafolio: {portfolio_id_matches if portfolio_id_matches else 'NINGUNA'}")
print("- Conclusion: el ejemplo visible del enunciado es ilustrativo y no constituye por si solo un dataset ejecutable de punta a punta.")

# ============================================================
# VERSION DEMO CONSISTENTE PARA VALIDAR LA LOGICA
# ============================================================

usuarios_demo = pd.DataFrame({
    "id_usuario": [123455, 123456],
    "email": ["analista@insightswm.com", "inversiones@insightswm.com"],
    "id_asesor": [9876, 9875],
    "perfil_riesgo": [5, 7],
})

portafolios_demo = pd.DataFrame({
    "id_portafolio": [45678, 45789],
    "id_usuario": [123455, 123456],
    "tipo_portafolio": ["acciones", "bonos"],
})

asesores_demo = pd.DataFrame({
    "id_asesor": [9876, 9875],
    "email": ["analista@insightswm.com", "insightswm@gmail.com"],
    "num_clientes": [3, 12],
})

balances_demo = pd.DataFrame({
    "id_portafolio": [45678, 45678, 45789, 45789],
    "fecha": pd.to_datetime([
        "2022-10-12",
        "2022-10-18",
        "2022-10-12",
        "2022-10-18",
    ]),
    "balance": [1500.15, 1490.00, 54300.60, 54420.20],
})

show_df("Usuarios - demo consistente", usuarios_demo)
show_df("Portafolios - demo consistente", portafolios_demo)
show_df("Asesores - demo consistente", asesores_demo)
show_df("Balances - demo consistente", balances_demo)

# Guardar tablas demo
usuarios_demo.to_csv(EJ1_OUTPUT_DIR / "usuarios_demo.csv", index=False)
portafolios_demo.to_csv(EJ1_OUTPUT_DIR / "portafolios_demo.csv", index=False)
asesores_demo.to_csv(EJ1_OUTPUT_DIR / "asesores_demo.csv", index=False)
balances_demo.to_csv(EJ1_OUTPUT_DIR / "balances_demo.csv", index=False)

# ============================================================
# PARAMETROS DE VALIDACION
# ============================================================

TARGET_ADVISOR_EMAIL = "insightswm@gmail.com"
TARGET_PORTFOLIO_TYPE = "bonos"
START_DATE = "2022-10-01"
END_DATE = "2022-10-31"

print("\nParametros de validacion:")
print(f"- email asesor objetivo: {TARGET_ADVISOR_EMAIL}")
print(f"- tipo de portafolio: {TARGET_PORTFOLIO_TYPE}")
print(f"- rango de fechas: {START_DATE} a {END_DATE}")

# ============================================================
# SECUENCIA LOGICA DEL EJERCICIO 1
# ============================================================

usuarios_portafolios = join_tables(
    from_table=portafolios_demo,
    to_table=usuarios_demo,
    fields="id_usuario"
)
show_df("Paso 1 - Usuarios_portafolios", usuarios_portafolios)

usuarios_portafolios = drop_fields(
    usuarios_portafolios,
    ["email", "perfil_riesgo"]
)
show_df("Paso 2 - Usuarios_portafolios sin email y perfil_riesgo", usuarios_portafolios)

usuarios_portafolios_asesores = join_tables(
    from_table=usuarios_portafolios,
    to_table=asesores_demo,
    fields="id_asesor"
)
show_df("Paso 3 - Usuarios_portafolios_asesores", usuarios_portafolios_asesores)

usuarios_portafolios_asesores = filter_table(
    usuarios_portafolios_asesores,
    "email",
    TARGET_ADVISOR_EMAIL
)
show_df("Paso 4 - Filtrado por email del asesor", usuarios_portafolios_asesores)

usuarios_portafolios_asesores = drop_fields(
    usuarios_portafolios_asesores,
    ["num_clientes"]
)
show_df("Paso 5 - Sin num_clientes", usuarios_portafolios_asesores)

tabla_final = join_tables(
    from_table=usuarios_portafolios_asesores,
    to_table=balances_demo,
    fields="id_portafolio"
)
show_df("Paso 6 - Join con balances", tabla_final)

tabla_final = filter_table(
    tabla_final,
    "tipo_portafolio",
    TARGET_PORTFOLIO_TYPE
)
show_df("Paso 7 - Filtrado por tipo_portafolio", tabla_final)

tabla_final = filter_table(
    tabla_final,
    "fecha",
    (START_DATE, END_DATE)
)

tabla_final = tabla_final[
    [
        "id_portafolio",
        "id_usuario",
        "id_asesor",
        "email",
        "tipo_portafolio",
        "fecha",
        "balance",
    ]
].sort_values(["fecha", "id_portafolio"]).reset_index(drop=True)

show_df("Paso 8 - Tabla final filtrada por rango de fechas", tabla_final)

# ============================================================
# RESPUESTA FINAL EN MARKDOWN
# ============================================================

respuesta_final_md = """# Ejercicio 1 - Manejo de Datos
**Prueba tecnica Insights**  
**Santiago Osorio Gomez**

## Objetivo
Construir una secuencia de funciones `join`, `filter` y `drop` que permita consultar el valor de los balances para un rango de fechas y segun tipo de portafolio, de los portafolios pertenecientes a usuarios cuyo asesor sea `insightswm@gmail.com`.

## Secuencia propuesta
1. `join(Portafolios, Usuarios, id_usuario, Usuarios_portafolios)`
2. `drop(Usuarios_portafolios, email & perfil_riesgo)`
3. `join(Usuarios_portafolios, Asesores, id_asesor, Usuarios_portafolios_asesores)`
4. `filter(Usuarios_portafolios_asesores, email, "insightswm@gmail.com")`
5. `drop(Usuarios_portafolios_asesores, num_clientes)`
6. `join(Usuarios_portafolios_asesores, Balances, id_portafolio, Tabla_final)`
7. `filter(Tabla_final, tipo_portafolio, <tipo_portafolio>)`
8. `filter(Tabla_final, fecha, [<fecha_inicio>, <fecha_fin>])`

## Justificacion
La logica de la solucion consiste en:
- relacionar primero los portafolios con sus usuarios mediante `id_usuario`;
- eliminar campos que no son necesarios para la consulta final;
- incorporar la informacion de asesores mediante `id_asesor`;
- filtrar unicamente los registros asociados al asesor `insightswm@gmail.com`;
- unir esa tabla resultante con los balances mediante `id_portafolio`;
- y finalmente aplicar los filtros de tipo de portafolio y rango de fechas para obtener la consulta requerida.

## Validacion en Python
Para verificar la logica, implemente una version reproducible en Python y ejecute una prueba con un caso demo consistente.

Caso de validacion usado:
- `tipo_portafolio = "bonos"`
- `fecha_inicio = "2022-10-01"`
- `fecha_fin = "2022-10-31"`

Resultado del caso demo:
- se identifico el portafolio `45789`;
- asociado al usuario `123456`;
- con asesor `insightswm@gmail.com`;
- y se obtuvieron los balances del rango consultado para las fechas `2022-10-12` y `2022-10-18`.

## Nota metodologica
Los registros visibles en el enunciado cumplen una funcion ilustrativa de la estructura de las tablas y de sus relaciones.
Para validar la secuencia completa en Python, construi una version demo consistente, manteniendo la misma logica relacional entre `Usuarios`, `Portafolios`, `Asesores` y `Balances`.
"""

# ============================================================
# GUARDAR RESULTADOS FINALES
# ============================================================

tabla_final.to_csv(EJ1_OUTPUT_DIR / "ejercicio_1_resultado_demo.csv", index=False)

with open(EJ1_OUTPUT_DIR / "ejercicio_1_respuesta_final.txt", "w", encoding="utf-8") as f:
    f.write(respuesta_final_md)

with open(DOCS_DIR / "ejercicio_1_respuesta_final.md", "w", encoding="utf-8") as f:
    f.write(respuesta_final_md)

print("\n" + "=" * 80)
print("RESPUESTA FINAL EN MARKDOWN")
print("=" * 80)
print(respuesta_final_md)

print("\nArchivos finales generados:")
print(f"- {EJ1_OUTPUT_DIR / 'usuarios_literal.csv'}")
print(f"- {EJ1_OUTPUT_DIR / 'portafolios_literal.csv'}")
print(f"- {EJ1_OUTPUT_DIR / 'asesores_literal.csv'}")
print(f"- {EJ1_OUTPUT_DIR / 'balances_literal.csv'}")
print(f"- {EJ1_OUTPUT_DIR / 'usuarios_demo.csv'}")
print(f"- {EJ1_OUTPUT_DIR / 'portafolios_demo.csv'}")
print(f"- {EJ1_OUTPUT_DIR / 'asesores_demo.csv'}")
print(f"- {EJ1_OUTPUT_DIR / 'balances_demo.csv'}")
print(f"- {EJ1_OUTPUT_DIR / 'ejercicio_1_resultado_demo.csv'}")
print(f"- {EJ1_OUTPUT_DIR / 'ejercicio_1_respuesta_final.txt'}")
print(f"- {DOCS_DIR / 'ejercicio_1_respuesta_final.md'}")

Raiz del proyecto detectada: /Users/santiago.os/Documents/prueba_insights
Directorio de datos: /Users/santiago.os/Documents/prueba_insights/data
Directorio docs: /Users/santiago.os/Documents/prueba_insights/docs
Directorio outputs ejercicio 1: /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_1

Verificacion de archivos reales del proyecto:
- instrucciones_pdf: OK -> /Users/santiago.os/Documents/prueba_insights/data/instrucciones.pdf
- prueba_xlsx: OK -> /Users/santiago.os/Documents/prueba_insights/data/prueba.xlsx
- withdrawals_xlsx: OK -> /Users/santiago.os/Documents/prueba_insights/data/withdrawals.xlsx

Usuarios - version literal PDF
------------------------------


,id_usuario,email,id_asesor,perfil_riesgo
0,123455,analista@insightswm.com,#1358,5
1,123456,inversiones@insightswm.com,#135813,7



Portafolios - version literal PDF
---------------------------------


,id_portafolio,id_usuario,tipo_portafolio
0,45678,123455,acciones
1,45789,123456,bonos



Asesores - version literal PDF
------------------------------


...


Paso 7 - Filtrado por tipo_portafolio
-------------------------------------


,id_portafolio,id_usuario,tipo_portafolio,id_asesor,email,fecha,balance
0,45789,123456,bonos,9875,insightswm@gmail.com,2022-10-12,54300.6
1,45789,123456,bonos,9875,insightswm@gmail.com,2022-10-18,54420.2



Paso 8 - Tabla final filtrada por rango de fechas
-------------------------------------------------


,id_portafolio,id_usuario,id_asesor,email,tipo_portafolio,fecha,balance
0,45789,123456,9875,insightswm@gmail.com,bonos,2022-10-12,54300.6
1,45789,123456,9875,insightswm@gmail.com,bonos,2022-10-18,54420.2



RESPUESTA FINAL EN MARKDOWN
# Ejercicio 1 - Manejo de Datos
**Prueba tecnica Insights**  
**Santiago Osorio Gomez**

## Objetivo
Construir una secuencia de funciones `join`, `filter` y `drop` que permita consultar el valor de los balances para un rango de fechas y segun tipo de portafolio, de los portafolios pertenecientes a usuarios cuyo asesor sea `insightswm@gmail.com`.

## Secuencia propuesta
1. `join(Portafolios, Usuarios, id_usuario, Usuarios_portafolios)`
2. `drop(Usuarios_portafolios, email & perfil_riesgo)`
3. `join(Usuarios_portafolios, Asesores, id_asesor, Usuarios_portafolios_asesores)`
4. `filter(Usuarios_portafolios_asesores, email, "insightswm@gmail.com")`
5. `drop(Usuarios_portafolios_asesores, num_clientes)`
6. `join(Usuarios_portafolios_asesores, Balances, id_portafolio, Tabla_final)`
7. `filter(Tabla_final, tipo_portafolio, <tipo_portafolio>)`
8. `filter(Tabla_final, fecha, [<fecha_inicio>, <fecha_fin>])`

## Justificacion
La logica de la solucion consiste en:
- rel

# Ejercicio 1 - Manejo de Datos
**Prueba tecnica Insights**  
**Santiago Osorio Gomez**

## Objetivo
Construir una secuencia de funciones `join`, `filter` y `drop` que permita consultar el valor de los balances para un rango de fechas y segun tipo de portafolio, de los portafolios pertenecientes a usuarios cuyo asesor sea `insightswm@gmail.com`.

## Secuencia propuesta
1. `join(Portafolios, Usuarios, id_usuario, Usuarios_portafolios)`
2. `drop(Usuarios_portafolios, email & perfil_riesgo)`
3. `join(Usuarios_portafolios, Asesores, id_asesor, Usuarios_portafolios_asesores)`
4. `filter(Usuarios_portafolios_asesores, email, "insightswm@gmail.com")`
5. `drop(Usuarios_portafolios_asesores, num_clientes)`
6. `join(Usuarios_portafolios_asesores, Balances, id_portafolio, Tabla_final)`
7. `filter(Tabla_final, tipo_portafolio, <tipo_portafolio>)`
8. `filter(Tabla_final, fecha, [<fecha_inicio>, <fecha_fin>])`

## Justificacion
La logica de la solucion consiste en:
- relacionar primero los portafolios con sus usuarios mediante `id_usuario`;
- eliminar campos que no son necesarios para la consulta final;
- incorporar la informacion de asesores mediante `id_asesor`;
- filtrar unicamente los registros asociados al asesor `insightswm@gmail.com`;
- unir esa tabla resultante con los balances mediante `id_portafolio`;
- y finalmente aplicar los filtros de tipo de portafolio y rango de fechas para obtener la consulta requerida.

## Validacion en Python
Para verificar la logica, implemente una version reproducible en Python y ejecute una prueba con un caso demo consistente.

Caso de validacion usado:
- `tipo_portafolio = "bonos"`
- `fecha_inicio = "2022-10-01"`
- `fecha_fin = "2022-10-31"`

Resultado del caso demo:
- se identifico el portafolio `45789`;
- asociado al usuario `123456`;
- con asesor `insightswm@gmail.com`;
- y se obtuvieron los balances del rango consultado para las fechas `2022-10-12` y `2022-10-18`.

## Nota metodologica
Los registros visibles en el enunciado cumplen una funcion ilustrativa de la estructura de las tablas y de sus relaciones.
Para validar la secuencia completa en Python, construi una version demo consistente, manteniendo la misma logica relacional entre `Usuarios`, `Portafolios`, `Asesores` y `Balances`.